# Open-Meteo Data Inventory

Goal: Explore Open-Meteo as a possible weather data source.

No cleaning, feature engineering, SQL import, or modeling in this notebook.

In [1]:
from pathlib import Path

RAW_DIR = Path("../data/raw/open_meteo")
RAW_DIR.mkdir(parents=True, exist_ok=True)

RAW_DIR

PosixPath('../data/raw/open_meteo')

In [2]:
import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry

from pathlib import Path

In [3]:
RAW_DIR = Path("../data/raw/open_meteo")
CACHE_DIR = Path("../.cache/open_meteo")

RAW_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

RAW_DIR, CACHE_DIR

(PosixPath('../data/raw/open_meteo'), PosixPath('../.cache/open_meteo'))

In [4]:
# Setup Open-Meteo API client with local cache and retry behavior

cache_session = requests_cache.CachedSession(
    CACHE_DIR / "open_meteo_cache",
    expire_after=-1
)

retry_session = retry(
    cache_session,
    retries=5,
    backoff_factor=0.2
)

openmeteo = openmeteo_requests.Client(session=retry_session)

In [5]:
# Test query: Berlin weather data
# This is only a raw API snapshot test, not final project data.

url = "https://archive-api.open-meteo.com/v1/archive"

params = {
    "latitude": 52.52,
    "longitude": 13.41,
    "start_date": "2026-04-18",
    "end_date": "2026-05-02",
    "hourly": [
        "temperature_2m",
        "precipitation",
        "apparent_temperature",
        "wind_speed_10m",
        "wind_direction_10m",
    ],
}

responses = openmeteo.weather_api(url, params=params)

response = responses[0]

print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

Coordinates: 52.5483283996582°N 13.407821655273438°E
Elevation: 38.0 m asl
Timezone difference to GMT+0: 0s


In [6]:
# Convert hourly API response to pandas DataFrame

hourly = response.Hourly()

hourly_data = {
    "date": pd.date_range(
        start=pd.to_datetime(hourly.Time(), unit="s", utc=True),
        end=pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True),
        freq=pd.Timedelta(seconds=hourly.Interval()),
        inclusive="left",
    )
}

variables = params["hourly"]

for i, variable in enumerate(variables):
    hourly_data[variable] = hourly.Variables(i).ValuesAsNumpy()

hourly_dataframe = pd.DataFrame(hourly_data)

hourly_dataframe.head()

,date,temperature_2m,precipitation,apparent_temperature,wind_speed_10m,wind_direction_10m
0,2026-04-18 00:00:00+00:00,10.90,0.0,7.918768,8.375320,81.347549
1,2026-04-18 01:00:00+00:00,10.00,0.0,6.968658,8.477216,93.652153
2,2026-04-18 02:00:00+00:00,9.15,0.0,6.146377,7.695920,100.784256
3,2026-04-18 03:00:00+00:00,8.50,0.0,5.654647,6.379216,106.389618
4,2026-04-18 04:00:00+00:00,7.80,0.0,4.925759,6.462476,102.875008


In [7]:
hourly_dataframe.shape

(360, 6)

In [8]:
hourly_dataframe.info()

<class 'pandas.DataFrame'>
RangeIndex: 360 entries, 0 to 359
Data columns (total 6 columns):
 #   Column                Non-Null Count  Dtype             
---  ------                --------------  -----             
 0   date                  360 non-null    datetime64[s, UTC]
 1   temperature_2m        360 non-null    float32           
 2   precipitation         360 non-null    float32           
 3   apparent_temperature  360 non-null    float32           
 4   wind_speed_10m        360 non-null    float32           
 5   wind_direction_10m    360 non-null    float32           
dtypes: datetime64[s, UTC](1), float32(5)
memory usage: 10.0 KB


In [9]:
hourly_dataframe.isna().sum()

date                    0
temperature_2m          0
precipitation           0
apparent_temperature    0
wind_speed_10m          0
wind_direction_10m      0
dtype: int64

In [10]:
hourly_dataframe.describe()

,temperature_2m,precipitation,apparent_temperature,wind_speed_10m,wind_direction_10m
count,360.000000,360.000000,360.000000,360.000000,360.000000
mean,11.185555,0.100000,7.956129,10.796411,192.961548
std,5.517993,0.525929,5.928986,4.743432,120.570190
min,1.450000,0.000000,-1.904916,0.000000,1.684657
25%,6.737500,0.000000,3.046565,7.019403,58.599541
50%,10.900000,0.000000,7.742558,10.867249,255.791573
75%,14.362500,0.000000,11.567737,13.538322,296.299721
max,26.450001,4.800000,23.594646,27.360592,360.000000


In [11]:
# Save local raw CSV snapshot

snapshot_name = "berlin_2026-04-18_2026-05-02_hourly.csv"
csv_path = RAW_DIR / snapshot_name

hourly_dataframe.to_csv(csv_path, index=False)

csv_path

PosixPath('../data/raw/open_meteo/berlin_2026-04-18_2026-05-02_hourly.csv')

In [12]:
# Save metadata for reproducibility

import json

metadata = {
    "provider": "Open-Meteo",
    "api": "Historical Weather API",
    "url": url,
    "latitude_requested": params["latitude"],
    "longitude_requested": params["longitude"],
    "latitude_returned": response.Latitude(),
    "longitude_returned": response.Longitude(),
    "elevation": response.Elevation(),
    "utc_offset_seconds": response.UtcOffsetSeconds(),
    "start_date": params["start_date"],
    "end_date": params["end_date"],
    "frequency": "hourly",
    "variables": params["hourly"],
    "local_csv_file": snapshot_name,
    "note": "Local raw API snapshot for data inventory. Not committed to Git.",
}

metadata_path = RAW_DIR / "berlin_2026-04-18_2026-05-02_metadata.json"

with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=2)

metadata_path

PosixPath('../data/raw/open_meteo/berlin_2026-04-18_2026-05-02_metadata.json')